In [ ]:
import torch
from torch import nn
from torch.nn import Module
from torchvision.datasets import mnist
from torch.nn import CrossEntropyLoss
from torch.optim import SGD
from torch.utils.data import DataLoader
from torchvision.transforms import ToTensor
import numpy as np
import os
import copy

class Model(Module):
    def __init__(self):
        super(Model,self).__init__()
        self.conv1=nn.Conv2d(1,6,5)
        self.relu1=nn.ReLU()
        self.maxpool1=nn.MaxPool2d(2)
        self.conv2=nn.Conv2d(6,16,5)
        self.relu2=nn.ReLU()
        self.maxpool2=nn.MaxPool2d(2)
        self.fc1=nn.Linear(256,120)
        self.relu3=nn.ReLU()
        self.fc2=nn.Linear(120,84)
        self.relu4=nn.ReLU()
        self.fc3=nn.Linear(84,10)

    def forward(self, x):
        y = self.conv1(x)
        y = self.relu1(y)
        y = self.maxpool1(y)
        y = self.conv2(y)
        y = self.relu2(y)
        y = self.maxpool2(y)
        y = y.view(y.shape[0], -1)
        y = self.fc1(y)
        y = self.relu3(y)
        y = self.fc2(y)
        y = self.relu4(y)
        y = self.fc3(y)
        return y

device='cpu'
batch_size=256
train_dataset=mnist.MNIST(root='./train', train=True, transform=ToTensor(), download=True)
test_dataset=mnist.MNIST(root='./test', train=False, transform=ToTensor(), download=True)
train_loader=DataLoader(train_dataset, batch_size=batch_size)
test_loader=DataLoader(test_dataset, batch_size=batch_size)
model=Model().to(device)
sgd=SGD(model.parameters(), lr=1e-1) #gradient descent
loss_fn=CrossEntropyLoss() #loss function
all_epoch=10

def measure_acc(model):
  all_correct_number=0
  all_sample_number=0
  model.eval()
  for idx,(test_x, test_label) in enumerate (test_loader):
    test_x = test_x.to(device)
    test_label=test_label.to(device)
    predict_y=model(test_x.float()).detach()
    predict_y=torch.argmax(predict_y, dim=-1)
    current_correct_number=predict_y==test_label
    all_correct_number += current_correct_number.sum().item()
    all_sample_number += test_x.shape[0]
  acc= all_correct_number / all_sample_number
  return(acc)

for current_epoch in range(all_epoch):
  model.train()
  for idx, (train_x,train_label) in enumerate (train_loader):
    train_x=train_x.to(device)
    train_label=train_label.to(device)
    sgd.zero_grad()                             #clear gradients
    predict_y=model(train_x.float())            #predict result
    loss=loss_fn(predict_y,train_label.long())  #mark/compute loss
    loss.backward()                             #get gradients
    sgd.step()                                  #write weights
  acc=measure_acc(model)

qm8=copy.deepcopy(model)
with torch.no_grad():
  for layer in [qm8.conv1, qm8.conv2, qm8.fc1, qm8.fc2, qm8.fc3]:
    w=layer.weight
    scale=w.abs().max()/127
    w_int=torch.round(w/scale)
    layer.weight.copy_(w_int)

qm4=copy.deepcopy(model)
with torch.no_grad():
  for layer in [qm4.conv1, qm4.conv2, qm4.fc1, qm4.fc2, qm4.fc3]:
    w=layer.weight
    scale=w.abs().max()/7
    w_int=torch.round(w/scale)
    layer.weight.copy_(w_int)

print('FP32',measure_acc(model))
print('INT8',measure_acc(qm8))
print('INT4',measure_acc(qm4))

print(model.conv1.weight)
print(qm8.conv1.weight)
print(qm4.conv1.weight)


print('finish training')

FP32 0.9842
INT8 0.9846
INT4 0.9843
Parameter containing:
tensor([[[[-0.0824,  0.0279,  0.1246, -0.1621, -0.2231],
          [-0.0631,  0.1896,  0.0199, -0.1905, -0.1579],
          [-0.2736, -0.0156,  0.2782,  0.2389,  0.4018],
          [-0.1820,  0.1206,  0.4263,  0.4759,  0.5326],
          [ 0.0365,  0.3709,  0.2057,  0.1111,  0.3691]]],


        [[[ 0.3443,  0.4261,  0.1672,  0.1855,  0.1282],
          [ 0.1064,  0.1210,  0.3302,  0.4641,  0.2559],
          [-0.2257,  0.1108, -0.0564,  0.1152,  0.2815],
          [-0.1598, -0.1008, -0.2955, -0.2440, -0.0535],
          [-0.3579, -0.3958, -0.3803, -0.2871, -0.2939]]],


        [[[ 0.0071, -0.1644,  0.3027,  0.3678,  0.2359],
          [-0.3080, -0.2773,  0.1752,  0.4886,  0.6316],
          [-0.1929, -0.2905,  0.0935,  0.6719,  0.3849],
          [-0.1787, -0.1684,  0.0576,  0.7513,  0.3068],
          [ 0.0277,  0.0179,  0.2788,  0.5507,  0.4162]]],


        [[[ 0.4152,  0.3406,  0.4612, -0.0501, -0.3229],
          [ 0.3003

In [ ]:
import torch
from torch import nn
from torch.nn import Module
from torchvision.datasets import mnist
from torch.nn import CrossEntropyLoss
from torch.optim import SGD
from torch.utils.data import DataLoader
from torchvision.transforms import ToTensor
import numpy as np
import os
import copy
import torch.nn.functional as F


class Model(Module):
    def __init__(self):
        super(Model,self).__init__()
        self.conv1=nn.Conv2d(1,6,5)
        self.relu1=nn.ReLU()
        self.maxpool1=nn.MaxPool2d(2)
        self.conv2=nn.Conv2d(6,16,5)
        self.relu2=nn.ReLU()
        self.maxpool2=nn.MaxPool2d(2)
        self.fc1=nn.Linear(256,120)
        self.relu3=nn.ReLU()
        self.fc2=nn.Linear(120,84)
        self.relu4=nn.ReLU()
        self.fc3=nn.Linear(84,10)

    def forward(self, x):
        y = self.conv1(x)
        y = self.relu1(y)
        y = self.maxpool1(y)
        y = self.conv2(y)
        y = self.relu2(y)
        y = self.maxpool2(y)
        y = y.view(y.shape[0], -1)
        y = self.fc1(y)
        y = self.relu3(y)
        y = self.fc2(y)
        y = self.relu4(y)
        y = self.fc3(y)
        return y

def fake_quant(w, qmax):
  scale = w.detach().abs().max()/qmax #maybe 7 or 127
  w_int = torch.round(w/scale)
  w_back = w_int*scale
  wq = w+(w_back-w).detach()
  return wq

class Model_Q(Module):
    def __init__(self,qmax):
        super(Model_Q,self).__init__()
        self.qmax = qmax
        self.conv1=nn.Conv2d(1,6,5)
        self.relu1=nn.ReLU()
        self.maxpool1=nn.MaxPool2d(2)
        self.conv2=nn.Conv2d(6,16,5)
        self.relu2=nn.ReLU()
        self.maxpool2=nn.MaxPool2d(2)
        self.fc1=nn.Linear(256,120)
        self.relu3=nn.ReLU()
        self.fc2=nn.Linear(120,84)
        self.relu4=nn.ReLU()
        self.fc3=nn.Linear(84,10)

    def forward(self, x):
        q = self.qmax
        y = F.conv2d(x, fake_quant(self.conv1.weight, q), self.conv1.bias)
        y = self.relu1(y)
        y = self.maxpool1(y)
        y = F.conv2d(y, fake_quant(self.conv2.weight, q), self.conv2.bias)
        y = self.relu2(y)
        y = self.maxpool2(y)
        y = y.view(y.shape[0], -1)
        y = F.linear(y, fake_quant(self.fc1.weight, q), self.fc1.bias)
        y = self.relu3(y)
        y = F.linear(y, fake_quant(self.fc2.weight, q), self.fc2.bias)
        y = self.relu4(y)
        y = F.linear(y, fake_quant(self.fc3.weight, q), self.fc3.bias)
        return y

device='cpu'
batch_size=256
train_dataset=mnist.MNIST(root='./train', train=True, transform=ToTensor(), download=True)
test_dataset=mnist.MNIST(root='./test', train=False, transform=ToTensor(), download=True)
train_loader=DataLoader(train_dataset, batch_size=batch_size)
test_loader=DataLoader(test_dataset, batch_size=batch_size)
model=Model().to(device)
sgd=SGD(model.parameters(), lr=1e-1) #gradient descent
loss_fn=CrossEntropyLoss() #loss function
all_epoch=10

def measure_acc(model): #the function that measure accuracy
  all_correct_number=0
  all_sample_number=0
  model.eval()
  for idx,(test_x, test_label) in enumerate (test_loader):
    test_x = test_x.to(device)
    test_label=test_label.to(device)
    predict_y=model(test_x.float()).detach()
    predict_y=torch.argmax(predict_y, dim=-1)
    current_correct_number=predict_y==test_label
    all_correct_number += current_correct_number.sum().item()
    all_sample_number += test_x.shape[0]
  acc= all_correct_number / all_sample_number
  return(acc)


for current_epoch in range(all_epoch):
  model.train()
  for idx, (train_x,train_label) in enumerate (train_loader):
    train_x=train_x.to(device)
    train_label=train_label.to(device)
    sgd.zero_grad()                             #clear gradients
    predict_y=model(train_x.float())            #predict result
    loss=loss_fn(predict_y,train_label.long())  #mark/compute loss
    loss.backward()                             #get gradients
    sgd.step()                                  #write weights
  acc=measure_acc(model)


qm4=copy.deepcopy(model) #PTQ
with torch.no_grad():
  for layer in [qm4.conv1, qm4.conv2, qm4.fc1, qm4.fc2, qm4.fc3]:
    w=layer.weight
    scale=w.abs().max()/7
    w_int=torch.round(w/scale)
    layer.weight.copy_(w_int)

qat4 = Model_Q(7) #QAT
qat4.load_state_dict(model.state_dict())
sgd_q = SGD(qat4.parameters(), lr=0.01)       # ← new：qat4 optimazor
for current_epoch in range(5):                # ← 5 epoch
  qat4.train()                                # ← Replace model to qat4
  for idx, (train_x,train_label) in enumerate (train_loader):
    train_x=train_x.to(device)
    train_label=train_label.to(device)
    sgd_q.zero_grad()                         # ← Replace sgd to sgd_q
    predict_y=qat4(train_x.float())           # ← Replace model to qat4
    loss=loss_fn(predict_y,train_label.long())
    loss.backward()
    sgd_q.step()                              # ← Replace sgd to sgd_q
  acc=measure_acc(qat4)                       # ← Replace model to qat4

qat4_int = Model()                                 # ← build a new model
qat4_int.load_state_dict(qat4.state_dict())         # ← put our trained QAT weight into new model
with torch.no_grad():
  for layer in [qat4_int.conv1, qat4_int.conv2, qat4_int.fc1, qat4_int.fc2, qat4_int.fc3]:
    w = layer.weight
    scale = w.abs().max()/7
    w_int = torch.round(w/scale)
    layer.weight.copy_(w_int)

print('FP32:           ', measure_acc(model))
print('PTQ INT4 (integer):', measure_acc(qm4))
print('QAT INT4 (fake_quant):', measure_acc(qat4))
print('QAT INT4 (integer):', measure_acc(qat4_int))

print(qat4.conv1.weight)
print(qat4_int.conv1.weight)
print('finish training')

FP32:            0.9831
PTQ INT4 (integer): 0.9829
QAT INT4 (fake_quant): 0.987
QAT INT4 (integer): 0.9867
Parameter containing:
tensor([[[[ 0.2569,  0.0856,  0.2098, -0.2622, -0.4895],
          [ 0.5086,  0.5900,  0.1375,  0.1388,  0.0374],
          [ 0.5080,  0.7199,  0.7903,  0.5605,  0.3439],
          [-0.0668,  0.4043,  0.4100,  0.4641,  0.4772],
          [-0.3539, -0.1440, -0.0107,  0.0661, -0.0131]]],


        [[[ 0.1578,  0.2070, -0.0937, -0.1530,  0.1779],
          [ 0.1201,  0.2461, -0.0810,  0.0391,  0.0514],
          [-0.1039,  0.2335,  0.1042,  0.0706, -0.0356],
          [ 0.1571,  0.3239,  0.0392,  0.0120, -0.1998],
          [ 0.0436,  0.2592, -0.0365, -0.0828, -0.2302]]],


        [[[ 0.1203, -0.0610,  0.0114,  0.0336, -0.1489],
          [ 0.1133,  0.0933,  0.3034,  0.3328,  0.2955],
          [ 0.1982,  0.0947, -0.0341,  0.2699,  0.0760],
          [ 0.2909, -0.1763, -0.1759, -0.1100, -0.1287],
          [ 0.2094, -0.0544, -0.1126, -0.1762,  0.0238]]],


    